# CASE 02 KPI & Segmentation

유형은 2025년 20대·30대 순이동 부호로만 나눈다.

In [1]:
from pathlib import Path
import sys

CASE_NAME = "02_youth_migration_dynamics"
RAW_NAME = "2025_domestic_migration_statistics.xlsx"
candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "cases" / CASE_NAME,
]
CASE_DIR = next(
    (path.resolve() for path in candidates if (path / "data" / "raw" / RAW_NAME).exists()),
    None,
)
if CASE_DIR is None:
    raise FileNotFoundError(
        f"{RAW_NAME}를 찾지 못했습니다. 저장소 루트 또는 notebooks 폴더에서 실행하세요."
    )
sys.path.insert(0, str(CASE_DIR / "src"))

from constants import RAW_FILE_NAME
from data_preparation import load_and_prepare
from data_quality import run_quality_checks
from kpi_segmentation import run_kpi_segmentation
from parse_official_tables import verify_source_file
from statistical_analysis import run_statistical_analysis

RAW = CASE_DIR / "data" / "raw" / RAW_FILE_NAME
source = verify_source_file(RAW)
prepared = load_and_prepare(RAW)
tables = prepared.tables
print(source["sha256"])
print("stale sheet present:", tables.workbook["has_stale_monthly_sheet"])

kpis = run_kpi_segmentation(tables, prepared)
kpis.values

FE066C40AAE0AE5C34C67947B404DC8943405C9BF0A7952DA09B0CF26D901E9D
stale sheet present: True


,kpi,value,unit,sido
0,Total movers 2025,6117784,persons,NaN
1,YoY change in total movers 2025,-2.6,percent,NaN
2,"Youth movers 20-39, 2025",2761243,persons,NaN
3,Youth share of movers 2025,0.451347,share,NaN
4,Inter-sido movers 2025,2185913,persons,NaN
5,Capital net vs non-capital 2025,38465,persons,NaN
6,"Youth 20-39 net, top inflow sido",21053,persons,경기
7,Seoul youth 20-39 net 2025,17207,persons,NaN
8,Youth age bands used,20-24+25-29+30-34+35-39,definition,NaN


In [2]:
kpis.typology_summary

,typology,typology_ko,sido_count,youth_net_sum,total_net_sum,sidos
0,Early Career Magnet,초입 유입·후기 유출형,1,17207,-26769,서울
1,Dual Magnet,청년 유입형,5,39746,78800,"경기, 인천, 충북, 대전, 세종"
2,Family Settle,후기 정착형,5,-18235,-7723,"충남, 울산, 전남, 대구, 경남"
3,Youth Outflow,청년 유출형,6,-38718,-44308,"제주, 강원, 전북, 부산, 광주, 경북"


In [3]:
kpis.priority

,priority_group,sido,typology_ko,net_youth_20_39,net_20s,net_30s,net_total,is_capital
0,"Early-career inflow, later outflow",서울,초입 유입·후기 유출형,17207,35937,-18730,-26769,True
1,Youth inflow (20s and 30s),경기,청년 유입형,21053,7439,13614,32970,True
2,Youth inflow (20s and 30s),인천,청년 유입형,12472,5004,7468,32264,True
3,Youth inflow (20s and 30s),충북,청년 유입형,2543,450,2093,10789,False
4,"20s outflow, 30s inflow",충남,후기 정착형,292,-1133,1425,8266,False
5,"20s outflow, 30s inflow",울산,후기 정착형,-1112,-1154,42,-5474,False
6,"20s outflow, 30s inflow",전남,후기 정착형,-4197,-5094,897,1334,False
7,"20s outflow, 30s inflow",대구,후기 정착형,-4673,-4884,211,-4272,False
8,"20s outflow, 30s inflow",경남,후기 정착형,-8545,-9129,584,-7577,False
9,Youth net outflow,경북,청년 유출형,-10759,-8839,-1920,-9214,False


In [4]:
kpis.dictionary

,kpi,formula,grain,period,include,exclude
0,Total movers 2025,"Sheet 1 총이동 이동자수, 남녀전체","registered mover, calendar year 2025",2025,시도내 + 시도간,"해외 이동, 미등록 이동"
1,"Youth movers 20-39, 2025",Sheet 2 20-24+25-29+30-34+35-39 이동자수,registered mover by 5-year age,2025,"남녀전체, 시도내+시도간",19세; e-지방지표 19-39와 직접 대사하지 않음
2,Youth share of movers 2025,Youth movers 20-39 / Total movers,national calendar year,2025,Sheet 2 남녀전체,순이동 지표가 아님
3,Inter-sido movers 2025,Sheet 1 시도간 이동자수,registered mover crossing a sido boundary,2025,시도간만,시군구 내부 이동
4,Capital net vs non-capital 2025,전입(비수도권→수도권) - 전출(수도권→비수도권),capital region vs rest of country,2025,서울·인천·경기,수도권 내부 서울↔경기 흐름
5,"Youth 20-39 net, top inflow sido","Sheet 5 20-39 순이동 합, 최댓값 시도","sido, 2025, 남녀전체",2025,네 개 5세 구간 순이동 합,전입지×전출지 청년 OD (원본에 없음)
